# 1단계 원본 수집

수집 대상: https://www.inven.co.kr/board/maple/2304/20563

`collect_main()`은 커밋된 `data/raw/` 산출물이 이미 있으면 기존 파일을 읽고 새로 요청하거나 덮어쓰지 않는다. 재수집이 필요할 때는 원본 산출물을 의도적으로 제거한 뒤 실행해야 한다.


In [1]:
from pathlib import Path
import json
import re
import sys

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

raw_dir = project_root / 'data' / 'raw'
raw_dir

WindowsPath('C:/Users/my/Desktop/프로젝트/sunday-maple/data/raw')

In [2]:
from src.collect import collect_main

collect_main(str(raw_dir))

In [3]:
expected_files = [
    'inven_20563.html',
    'inven_20563_meta.json',
    'view_by_type.txt',
    'view_by_date.txt',
]

for name in expected_files:
    path = raw_dir / name
    text = path.read_text(encoding='utf-8')
    print(f'{name}: 존재={path.exists()} 글자수={len(text)} 바이트={path.stat().st_size}')

meta = json.loads((raw_dir / 'inven_20563_meta.json').read_text(encoding='utf-8'))
print('HTTP 상태 코드:', meta.get('status_code'))
print('robots.txt 허용 여부:', meta.get('robots_allowed'))


inven_20563.html: 존재=True 글자수=629378 바이트=698829
inven_20563_meta.json: 존재=True 글자수=417 바이트=426
view_by_type.txt: 존재=True 글자수=23476 바이트=31110
view_by_date.txt: 존재=True 글자수=43800 바이트=82038
HTTP 상태 코드: 200
robots.txt 허용 여부: True


In [4]:
for name in ['view_by_type.txt', 'view_by_date.txt']:
    text = (raw_dir / name).read_text(encoding='utf-8')
    date_count = len(re.findall(r'\d{4}\.\d{2}\.\d{2}', text))
    print(f'{name}: 날짜 패턴 수={date_count} 굵게표시 수={text.count("[[B]]")} 취소선표시 수={text.count("[[S]]")}')


view_by_type.txt: 날짜 패턴 수=1140 굵게표시 수=71 취소선표시 수=0
view_by_date.txt: 날짜 패턴 수=503 굵게표시 수=568 취소선표시 수=2


In [5]:
for name in ['view_by_type.txt', 'view_by_date.txt']:
    print(f'--- {name} 처음 30줄 ---')
    lines = (raw_dir / name).read_text(encoding='utf-8').splitlines()[:30]
    print('\n'.join(lines))


--- view_by_type.txt 처음 30줄 ---
- [HEXA 스탯] 메인 스탯이 5레벨 이상일 때, 메인 스탯의 강화 확률 20% 증가
2024.07.21
(+84일)2024.10.13
(+91일)2025.01.12
(+84일)2025.04.06
(+91일)2025.07.06
(+112일)2025.10.26
(+77일)2026.01.11
(+98일)2026.04.19
- 10성 이하에서 스타포스 강화 성공 시 1+1 강화
2018.02.18
(+98일)2018.05.27
(+49일)2018.07.15
(+140일)2018.12.02
(+63일)2019.02.03
(+98일)2019.05.12
(+49일)2019.06.30
(+63일)2019.09.01
(+119일)2019.12.29
(+98일)2020.04.05
(+77일)2020.06.21
(+42일)2020.08.02
(+77일)2020.10.18
(+126일)2021.02.21
(+63일)2021.04.25
(+49일)2021.06.13
(+49일)2021.08.01
(+49일)2021.09.19
(+49일)2021.11.07
(+154일)2022.04.10
--- view_by_date.txt 처음 30줄 ---
2017.03.12 (-)
주문의 흔적 강화 비용 30% 할인
(리부트, 리부트2월드 : 하루 종일 드롭률 2배)
돌발미션 일일 클리어 가능 횟수 2배(3회→6회)
돌발미션 클리어 경험치 2배
[[S]]2017.03.19[[/S]]
[[S]]아예 없음[[/S]]
2017.03.26 (도전! 메이플 레인보우!)
스타포스 5, 10, 15성에서 강화 시도 시 성공확률 100%
룬 경험치 버프 효과 +100%
룬 경험치 버프 지속시간 2배(3분→6분)
2017.04.02 (달콤한 질주 : 더 꿀)
아케인리버 1~6지역 일일 컨텐츠 획득 보상 2배
폴로와 프리토의 현상금 사냥 경험치 1.5배
2017.04.09 (달콤한 질주 : 더 꿀)
어빌리티 재설정 비용 50% 할인
버닝 필드 더 빠른

## 재현성 메모

- `data/raw/`는 커밋된 원본이며 이후 단계에서는 불변으로 취급한다.
- `save_raw_html()`은 HTML 파일이 이미 있으면 기존 메타데이터를 반환한다.
- `collect_main()`은 추출 텍스트 파일이 없을 때만 새로 쓴다.
- 커밋된 원본 파일이 있는 상태에서 01 노트북을 다시 실행해도 `git status data/raw/`는 비어 있어야 한다.
